# 임베딩 모델 비교: bge-m3 · KURE-v1 · CLOVA Studio

실전 2 연구계획서 E2(검색 방식·모델 비교) 사전 점검. **로컬 Mac에서만 실행**합니다 (카탈로그 데이터가 밖으로 나가지 않음).

| 모델 | 실행 위치 | 비고 |
|---|---|---|
| bge-m3 (`BAAI/bge-m3`) | 이 노트북 (로컬, Apple GPU) | 처음 실행 시 모델 약 2GB 내려받음 |
| KURE-v1 (`nlpai-lab/KURE-v1`) | 이 노트북 (로컬) | bge-m3를 한국어로 추가 학습한 모델 |
| CLOVA Studio v2 | API (`app/services/embedding.py`) | 분당 60건 제한 → 표 2,364개에 약 40분. `RUN_CLOVA = True`일 때만 |
| 키워드 (현재 검색) | `app/services/search.py` | 비교 기준 |

**실행 방법** (프로젝트 루트에서. 프로젝트 의존성·Docker 이미지는 바꾸지 않음)
```bash
uv run --with jupyterlab --with sentence-transformers jupyter lab notebooks/embedding_compare.ipynb
```

**평가 방식:** 일상어 질문 10개(`scripts/check_embedding.py`의 `PROBES`)로 전체 표를 검색해, 규칙으로 정한 관련 표(표명에 규칙 단어가 있는 표)가 Top-10에 몇 개 드는지, 첫 관련 표 순위를 봅니다. **골든셋이 아닌 규칙 기반 예비 점검**이라 정확도 주장에는 쓰지 않고, 정식 비교는 골든셋으로 하는 E2에서 합니다.

**결과 파일**
- 임베딩: `data/embeddings/{모델}__{문서구성}.npy` (data/는 커밋하지 않음, 다시 실행하면 재사용)
- 비교표: `eval/results/embedding_compare_{문서구성}_{날짜}.md / .csv`

In [ ]:
import os
import sys
import time
from datetime import datetime
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)  # .env, data/ 경로 기준
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import torch

from pipeline.load_catalog import read_catalog
from pipeline.search_docs import PRESETS, build
from scripts.check_embedding import PROBES, TOP_K, ranking_stats

DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
EMB_DIR = ROOT / "data" / "embeddings"
RESULTS = ROOT / "eval" / "results"
EMB_DIR.mkdir(parents=True, exist_ok=True)
print("ROOT", ROOT, "| device", DEVICE)

## 1. 설정

- `PRESET`: 검색 문서 구성 (`name` 조사명+표명 / `classes` +분류명 / `full` +분류 값·항목명·경로). 연구계획서 E3 필드 비교는 이 값만 바꿔 다시 실행합니다.
- clir-dolphin 모델은 입력 한도가 약 512토큰이라 `full`에서는 23% 표를 처리하지 못합니다 (`eval/results/embedding_check_20261030.md`).

In [ ]:
PRESET = "full"
LOCAL_MODELS = {"bge-m3": "BAAI/bge-m3", "KURE-v1": "nlpai-lab/KURE-v1"}
RUN_CLOVA = False          # True면 CLOVA v2로도 임베딩 (약 40분, 한 번 만들면 파일 재사용)
CLOVA_MODELS = ["v2"]
BATCH_SIZE = 16

## 2. 표 문서 만들기 (카탈로그 파일에서 바로, DB 불필요)

In [ ]:
tables = read_catalog()
ids = [t["tbl_id"] for t in tables]
names = [t["table_name"] for t in tables]
docs = [build(t, PRESET) for t in tables]

lengths = pd.Series([len(d) for d in docs])
print(f"표 {len(docs)}개 · 문서 구성 {PRESET}")
print(lengths.describe(percentiles=[0.5, 0.9]).round(0).to_string())
print("\n예시:", docs[ids.index("TX_11025_A020")][:300])

## 3. 로컬 모델로 임베딩 (파일이 있으면 재사용)

In [ ]:
from sentence_transformers import SentenceTransformer


def cache_path(model: str) -> Path:
    return EMB_DIR / f"{model}__{PRESET}.npy"


def load_cache(model: str) -> np.ndarray | None:
    path, id_path = cache_path(model), cache_path(model).with_suffix(".ids")
    if path.exists() and id_path.exists() and id_path.read_text().split() == ids:
        return np.load(path)
    return None


def save_cache(model: str, vectors: np.ndarray) -> None:
    np.save(cache_path(model), vectors)
    cache_path(model).with_suffix(".ids").write_text("\n".join(ids))


vectors: dict[str, np.ndarray] = {}
encoders: dict[str, SentenceTransformer] = {}
timings: dict[str, float] = {}
for label, hf_name in LOCAL_MODELS.items():
    encoders[label] = SentenceTransformer(hf_name, device=DEVICE)
    cached = load_cache(label)
    if cached is not None:
        vectors[label] = cached
        print(f"{label}: 저장된 임베딩 사용 {cached.shape}")
        continue
    start = time.perf_counter()
    vectors[label] = encoders[label].encode(
        docs, batch_size=BATCH_SIZE, normalize_embeddings=True, show_progress_bar=True
    )
    timings[label] = time.perf_counter() - start
    save_cache(label, vectors[label])
    print(f"{label}: {vectors[label].shape} · {timings[label]:.0f}초")

## 4. CLOVA Studio (선택, `RUN_CLOVA = True`일 때)

In [ ]:
from app.services import embedding as clova

for model in CLOVA_MODELS:
    label = f"clova-{model}"
    cached = load_cache(label)
    if cached is not None:
        vectors[label] = cached
        print(f"{label}: 저장된 임베딩 사용 {cached.shape}")
    elif RUN_CLOVA:
        start = time.perf_counter()
        raw = np.array(await clova.embed(docs, model), dtype=np.float32)
        vectors[label] = raw / np.linalg.norm(raw, axis=1, keepdims=True)
        timings[label] = time.perf_counter() - start
        save_cache(label, vectors[label])
        print(f"{label}: {vectors[label].shape} · {timings[label]:.0f}초")
    else:
        print(f"{label}: 건너뜀 (RUN_CLOVA = False, 저장된 임베딩 없음)")

## 5. 일상어 질문 10개로 비교

In [ ]:
import re

from app.services import search
from app.services import slots as slot_parser

keyword_tables = search.prepare(tables)


async def query_vector(label: str, question: str) -> np.ndarray:
    if label in encoders:
        return encoders[label].encode([question], normalize_embeddings=True)[0]
    raw = np.array((await clova.embed([question], label.removeprefix("clova-")))[0])
    return raw / np.linalg.norm(raw)


rows = []
for question, rule in PROBES:
    relevant = [bool(re.search(rule, n)) for n in names]
    methods = {
        "keyword": search.keyword_scores(
            keyword_tables, search.query_tokens(slot_parser.empty_slots(question))
        )
    }
    for label, matrix in vectors.items():
        methods[label] = (matrix @ await query_vector(label, question)).tolist()
    for method, scores in methods.items():
        stats = ranking_stats(scores, relevant)
        top = sorted(scores, reverse=True)
        top5 = sorted(range(len(scores)), key=lambda i: -scores[i])[:5]
        rows.append({
            "question": question,
            "rule": rule,
            "relevant_tables": sum(relevant),
            "method": method,
            f"hits@{TOP_K}": stats[f"hits@{TOP_K}"],
            "first_rank": stats["first_rank"],
            "score_top1": round(top[0], 3),
            "score_top10": round(top[TOP_K - 1], 3),
            "score_median": round(float(np.median(scores)), 3),
            "top5": " / ".join(names[i] for i in top5),
        })

result = pd.DataFrame(rows)
summary = result.groupby("method", sort=False).agg(
    **{
        f"Top-{TOP_K} 안 관련 표 (평균)": (f"hits@{TOP_K}", "mean"),
        "첫 관련 표가 1위인 질문": ("first_rank", lambda s: int((s == 1).sum())),
        "첫 관련 표 순위 (중앙값)": ("first_rank", "median"),
        "1위 점수 (평균)": ("score_top1", "mean"),
        "중앙값 점수 (평균)": ("score_median", "mean"),
    }
).round(2)
summary

In [ ]:
# 질문별: Top-10 안 관련 표 수 / 첫 관련 표 순위
pivot = result.pivot_table(
    index="question", columns="method", values=[f"hits@{TOP_K}", "first_rank"], sort=False
)
pivot

## 6. 질문별 상위 5개 표 (정성 확인)

In [ ]:
pd.set_option("display.max_colwidth", None)
result[["question", "method", "top5"]]

## 7. 결과 저장 (`eval/results/`)

In [ ]:
stem = RESULTS / f"embedding_compare_{PRESET}_{datetime.now():%Y%m%d}"
result.to_csv(stem.with_suffix(".csv"), index=False, encoding="utf-8-sig")
lines = [
    f"# 임베딩 모델 비교 ({PRESET}) {datetime.now():%Y-%m-%d %H:%M}",
    "",
    f"- 문서 구성: `{PRESET}` (pipeline/search_docs.py), 표 {len(docs)}개, "
    f"문서 길이 중앙값 {lengths.median():.0f}자",
    f"- 로컬 모델 실행: {DEVICE} · 임베딩 시간(초): "
    + (", ".join(f"{k} {v:.0f}" for k, v in timings.items()) or "저장된 임베딩 재사용"),
    "- 관련 표 = 표명에 규칙 단어가 있는 표. **골든셋이 아닌 규칙 기반 점검**이라 정확도 주장에는 쓰지 않는다.",
    "",
    "## 요약",
    "",
    "| 방식 | " + " | ".join(summary.columns) + " |",
    "|---|" + "---|" * len(summary.columns),
    *[f"| {m} | " + " | ".join(str(v) for v in row) + " |" for m, row in summary.iterrows()],
    "",
    "## 질문별 (Top-10 안 관련 표 수 / 첫 관련 표 순위)",
    "",
]
methods = list(dict.fromkeys(result["method"]))
lines.append("| 질문 | 관련 표 수 | " + " | ".join(methods) + " |")
lines.append("|---|---|" + "---|" * len(methods))
for question, group in result.groupby("question", sort=False):
    g = group.set_index("method")
    cells = [f"{g.loc[m, f'hits@{TOP_K}']} / {g.loc[m, 'first_rank']}" for m in methods]
    lines.append(f"| {question} | {g['relevant_tables'].iloc[0]} | " + " | ".join(cells) + " |")
stem.with_suffix(".md").write_text("\n".join(lines) + "\n", encoding="utf-8")
print("저장:", stem.with_suffix(".md"), stem.with_suffix(".csv"))